# Tater — Jupyter Annotation Example

This notebook demonstrates how to run a Tater annotation app inline in a Jupyter notebook.

**Workflow:**
1. Define a Pydantic schema
2. Call `tater.annotate()` — the app renders below the cell
3. Annotate documents in the embedded UI
4. Call `app.get_annotations()` in a later cell to retrieve results

## 1. Define the annotation schema

In [ ]:
from typing import Literal, Optional
from pydantic import BaseModel


class DocumentReview(BaseModel):
    sentiment: Optional[Literal["positive", "negative", "neutral"]] = None
    confidence: Optional[Literal["high", "medium", "low"]] = None
    notes: Optional[str] = None

## 2. Launch the annotation UI

Documents can be a file path **or** an in-memory list of dicts / DataFrame.
Annotations are auto-saved when a file path is used; pass ``annotations_path``
explicitly for in-memory documents if you want persistence.

Change `jupyter_mode` to `"tab"` to open in a new browser tab instead.

In [ ]:
import tater

# Option A — file path (annotations auto-saved to documents_inline_annotations.json)
app = tater.annotate(
    model=DocumentReview,
    documents="../data/documents_inline.json",
    title="Document Review",
    jupyter_mode="inline",
    port=8050,
)

# Option B — in-memory list of dicts
# docs = [
#     {"id": "doc_1", "text": "The product exceeded my expectations."},
#     {"id": "doc_2", "text": "Delivery was slow and packaging was damaged."},
# ]
# app = tater.annotate(
#     model=DocumentReview,
#     documents=docs,
#     annotations_path="my_annotations.json",  # required for auto-save with in-memory docs
#     jupyter_mode="inline",
#     port=8050,
# )

## 3. Retrieve annotations

Run this cell after annotating. Returns the latest auto-saved state as a
`{doc_id: annotation_dict}` mapping.

In [ ]:
results = app.get_annotations()
results

## 4. Convert to a DataFrame (optional)

In [ ]:
import pandas as pd

df = pd.DataFrame.from_dict(results, orient="index")
df.index.name = "doc_id"
df

---
## Advanced: manual widget configuration

Use `TaterApp` directly when you need custom widgets, hooks, or span annotation.

In [ ]:
from tater import TaterApp, widgets_from_model
from tater.widgets.segmented_control import SegmentedControlWidget
from tater.widgets.textarea import TextAreaWidget

widgets = [
    SegmentedControlWidget("sentiment", label="Sentiment"),
    SegmentedControlWidget("confidence", label="Confidence"),
    TextAreaWidget("notes", label="Notes", description="Optional free-text comments"),
]

app2 = TaterApp(
    schema_model=DocumentReview,
    title="Document Review (custom widgets)",
)
app2.load_documents("../data/documents_inline.json")
app2.set_annotation_widgets(widgets)
app2.run(jupyter_mode="inline", port=8051)